In [1]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# Establish path(s)
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from config.project_paths import (
    ROOT,
    DATA_RAW,
    DATA_PROCESSED,
    FIGURES,
    YEAR_MIN,
    YEAR_MAX,
    get_raw_data_path,
    get_processed_data_path,
    get_figure_path,
    validate_paths,
)

In [2]:
# Verify all paths
print("ROOT:", ROOT)
print("RAW DATA:", DATA_RAW)
print("PROCESSED DATA:", DATA_PROCESSED)
print("FIGURES:", FIGURES)

ROOT: /home/ggirelli/Documents/DataAnalysis/Projects/Capstone_1
RAW DATA: /home/ggirelli/Documents/DataAnalysis/Projects/Capstone_1/data/raw
PROCESSED DATA: /home/ggirelli/Documents/DataAnalysis/Projects/Capstone_1/data/processed
FIGURES: /home/ggirelli/Documents/DataAnalysis/Projects/Capstone_1/outputs/figures


---
**Load data set and test for errors**
---

In [3]:
# Load raw data
RAW_DEMOGRAPHIC_ASPECTS_FILE = get_raw_data_path("Demographic-aspects-2023.xlsx")
TABLE_1_9 = get_raw_data_path("Table-1.9-Relative-population-by-age-groups-in-percentage.xlsx")

validate_paths({
    "Demographic aspects": RAW_DEMOGRAPHIC_ASPECTS_FILE,
    "Table-1.9-Relative-population-by-age-groups-in-percentage":TABLE_1_9
})

In [4]:
# Load raw data
# Both sheets use row 1 (0-indexed) as the real header → header=1
df_main = pd.read_excel("/home/ggirelli/Documents/DataAnalysis/Datasets/Spreadsheets/Official_Open-Datasets_Aruba_CBS/Demographic-aspects-2023.xlsx", header=1)
df2_to_merge = pd.read_excel("/home/ggirelli/Documents/DataAnalysis/Datasets/Spreadsheets/Official_Open-Datasets_Aruba_CBS/Table-1.9-Relative-population-by-age-groups-in-percentage.xlsx", header=1)

In [5]:
# Clean df_main → extract total population per year
# Normalise the text column to avoid invisible-whitespace mismatches
df_main["Key Demographic aspects"] = (
    df_main["Key Demographic aspects"].astype(str).str.strip()
)

df_total_row = df_main[df_main["Key Demographic aspects"] == "Total population"]

In [6]:
df_total_row

,Key Demographic aspects,Unit,2015,2016,2017,2018,2019,2020,2021,2022,2023
2,Total population,Absolute,108635.0,108818.0,108651.0,109164.0,109241.0,107932.0,107468.0,107152.0,107566.0


In [7]:
# Melt from wide → long; years become rows
df_total_tidy = df_total_row.melt(
    id_vars=["Key Demographic aspects", "Unit"],
    var_name="Year",
    value_name="Total_Count"
)

In [8]:
# Keep only the two columns we need downstream
df_total_tidy = df_total_tidy[["Year", "Total_Count"]]
df_total_tidy["Year"] = df_total_tidy["Year"].astype(int)

print("── df_total_tidy ──")
print(df_total_tidy.to_string(index=False))

── df_total_tidy ──
 Year  Total_Count
 2015     108635.0
 2016     108818.0
 2017     108651.0
 2018     109164.0
 2019     109241.0
 2020     107932.0
 2021     107468.0
 2022     107152.0
 2023     107566.0


In [9]:
# Clean df_to_merge → age-group percentages per sex per year
# The raw sheet has a two-row header:
# Row 0 (after header=1 read): year numbers (2015 … 2023), every 3rd column
# Row 1 (actual data row 0):   sub-labels "Male", "Female", "Total"
# Strategy: rename ALL columns explicitly, then drop the sub-label row.

years = range(2015, 2024)

new_cols = ["Age"]
for y in years:
    new_cols += [f"Male_{y}", f"Female_{y}", f"Total_{y}"]  # 9 × 3 = 27 cols + Age = 28

df2_to_merge.columns = new_cols          # overwrite the messy auto-generated names
df2 = df2_to_mer.iloc[1:].copy()       # drop row 0 (the "Male / Female / Total" sub-header)

# Keep only the meaningful age-band rows; discard ratio/footnote rows
keep_ages = ["0 - 14", "15 - 59", "60 +", "Total"]
df2_to_merge["Age"] = df2_to_merge["Age"].astype(str).str.strip()
df_age = df2_to_merge[df2_to_merge["Age"].isin(keep_ages)].reset_index(drop=True)

print("\n── df_age (wide, cleaned) ──")
print(df_age.to_string(index=False))

NameError: name 'df2_raw' is not defined

In [ ]:
# Melt df_age → tidy long form
df_age_long = df_age.melt(
    id_vars=["Age"],
    var_name="Sex_Year",        # e.g. "Male_2015"
    value_name="Percentage"
)

In [ ]:
# Split "Male_2015" into two columns → Sex="Male", Year=2015
df_age_long[["Sex", "Year"]] = (
    df_age_long["Sex_Year"].str.rsplit("_", n=1, expand=True)
)
df_age_long["Year"] = df_age_long["Year"].astype(int)
df_age_long["Percentage"] = pd.to_numeric(df_age_long["Percentage"], errors="coerce")
df_age_long = df_age_long.drop(columns=["Sex_Year"]).reset_index(drop=True)

print("\n── df_age_long (first 12 rows) ──")
print(df_age_long.head(12).to_string(index=False))

In [ ]:
# Merge both DataFrames on Year
df_final = df_age_long.merge(df_total_tidy, on="Year", how="left")

# Bonus column: back-calculate absolute counts from % × total
df_final["Abs_Count"] = (
    df_final["Percentage"] / 100 * df_final["Total_Count"]
).round(0).astype("Int64")   # Int64 supports NaN natively

In [ ]:
# Tidy column order and sort
df_final = (
    df_final[["Year", "Age", "Sex", "Percentage", "Total_Count", "Abs_Count"]]
    .sort_values(["Year", "Age", "Sex"])
    .reset_index(drop=True)
)

In [ ]:
# Quick sanity check
# For Sex=="Total", the three age groups should sum to ~100 %
check = df_final[(df_final["Sex"] == "Total") & (df_final["Age"] != "Total")].groupby("Year")["Percentage"].sum()
     
print("\n── Column sum per year (Sex='Total', should be 100) ──")
print(check)

In [ ]:
# Quick starter plot
# Filter to totals only, exclude the grand-total "Total" age row
plot_df = df_final[
    (df_final["Sex"] == "Total") & (df_final["Age"] != "Total")
]

fig, ax = plt.subplots(figsize=(10, 5))
for age, grp in plot_df.groupby("Age"):
    ax.plot(grp["Year"], grp["Percentage"], label=age)

ax.set_title("Aruba population change by age group (2015–2023)")
ax.set_ylabel("% of total population")
ax.set_xlabel("Year")
ax.legend(title="Age group")
ax.grid(True, linestyle="--", alpha=0.5)
#plt.tight_layout()

plt.show()